In [ ]:
import requests
import time  # to add delay
import json
import os  # to create directories and rename files
from datetime import datetime, timedelta
from requests_oauthlib import OAuth1

In [ ]:
print(os.getcwd())

In [ ]:
os.chdir("Data collection")

In [ ]:
# tags

tags = ['sexism', 'disabled', 'transfeminism', 'genderqueer','qtpoc', 'people of color', 'justice', 'lgbt', 'sexuality','womens rights','gender', 'white feminism', 'blackwomen', 'sisterhood', 'sjw', 'animal rights', 'lgbtq', 'bisexual', 'asexual', 'identity', 'vegan', 'socialjustice', 'queer', 'blacklivesmatter', 'gender identity', 'patriarchy', 'social justice', 'girl power', 'smash the patriarchy', 'blackgirlmagic', 'ethnicity','intersectional feminism', 'asexuality', 'lgbtq community', 'mental illness', 'identity politics', 'intersectional', 'pro choice', 'trans women', 'abortion', 'fat acceptance', 'women of color', 'marxism','inclusion', 'resistance', 'representation matters', 'intersectional activism', 'transgender', 'wage gap', 'audre lorde', 'feminism', "women's rights", 'feminist blog', 'poc', 'misandry', 'the future is intersectional', 'kimberle crenshaw', 'bodily autonomy', 'oppression', 'socialism', 'lgbtq issues', 'intersectional feminist', 'revolution', 'sociology', 'bell hooks', 'gender equality', 'womanism', 'disability', 'trans rights', 'ableism', 'nonbinary', 'race', 'latina', 'imperialism', 'black lives matter', 'trans', 'radical feminism', 'non-binary', 'intersectional social justice', 'rape culture', 'intersectionalfeminism', 'diversity', 'privilege', 'womanist', 'capitalism', 'anarchism', 'woc', 'lgbtpride', 'liberation', "international women's day", 'black feminist thought', 'intersections', 'body positive', 'lgbt rights', 'womensrights', 'intersectionalfeminist', 'class', 'black feminist', 'black feminism', 'lgbtqia', 'discrimination', 'disabilities', 'pansexual', 'equal rights', 'equality', 'ability', 'blm', 'pride', 'transwomen', 'body positivity', 'lesbian', 'inclusive feminism', 'qpoc', 'solidarity', 'representation', 'women of colour', 'muslim', 'body image', 'autism', 'civil rights', 'inclusive', 'black women', 'sex positive', 'feminists', 'veganism', 'gay']


In [ ]:
len(tags)

In [ ]:
# define the tag and base_url

base_url = "https://api.tumblr.com/v2/tagged"
auth_list = [
    OAuth1(my_key_1, secret_key_1),
    OAuth1(my_key_2, secret_key_2),
    OAuth1(my_key_3, secret_key_3),
    OAuth1(my_key_4, secret_key_4),
    OAuth1(my_key_5, secret_key_5),
    OAuth1(my_key_6, secret_key_6),
    OAuth1(my_key_7, secret_key_7),
    OAuth1(my_key_8, secret_key_8),
    OAuth1(my_key_9, secret_key_9),
    OAuth1(my_key_10, secret_key_10),
    OAuth1(my_key_11, secret_key_11),
    OAuth1(my_key_12, secret_key_12),
    OAuth1(my_key_13, secret_key_13),
    OAuth1(my_key_14, secret_key_14),
    OAuth1(my_key_15, secret_key_15),
    OAuth1(my_key_16, secret_key_16),
    OAuth1(my_key_17, secret_key_17),
    OAuth1(my_key_18, secret_key_18),
    OAuth1(my_key_19, secret_key_19),
    OAuth1(my_key_20, secret_key_20),
    OAuth1(my_key_21, secret_key_21),
    OAuth1(my_key_22, secret_key_22),
    OAuth1(my_key_23, secret_key_23),
    OAuth1(my_key_24, secret_key_24),
    OAuth1(my_key_25, secret_key_25),
    OAuth1(my_key_26, secret_key_26),
    OAuth1(my_key_27, secret_key_27),
    OAuth1(my_key_28, secret_key_28)
    
]

# create a folder to store data if it doesn't exist
    
os.makedirs(data_folder, exist_ok=True)

# save data to a json file

def save_json(data, filename):
    """ Save data to a JSON file. """
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

# to resume progress if the script crashes        

def load_json(filename):
    """ Load data from a JSON file. """
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

# function to fetch posts for a specific time range

def fetch_posts_for_period(oldest_ts, latest_ts, temp_filename):
    """ Fetch posts within the given timestamp range, saving progress incrementally. """
    current_ts = latest_ts
    print(f"temp_filename: {temp_filename}")
    monthly_posts = load_json(temp_filename)  # Load any existing data
    current_auth_key = 0

    if monthly_posts:
        # resume from the last post's timestamp
        current_ts = min(post["timestamp"] for post in monthly_posts)
        print(f"Resuming from {datetime.utcfromtimestamp(current_ts)} ({current_ts})")

    while current_ts > oldest_ts:
        auth = auth_list[current_auth_key]
        params = {"tag": tag, "before": current_ts}
        response = requests.get(base_url, params=params, auth=auth)

        if response.status_code == 200:
            pass
        elif response.status_code == 429:
            print(f"Key expired. Current key index: {current_auth_key}")
            current_auth_key = (current_auth_key + 1) % len(auth_list)
            continue
        else:
            raise Exception(f"Request unsuccessful! Status code: {response.status_code}")

        posts = response.json().get("response", [])
        filtered_posts = [post for post in posts if (oldest_ts <= post["timestamp"] <= current_ts)]

        print(f"[{tag}] Fetched {len(filtered_posts)} posts - Current date: {datetime.utcfromtimestamp(current_ts)}")

        #if not filtered_posts:
            #break  # No more posts in this period

        monthly_posts.extend(filtered_posts)

        # save progress after each batch
        save_json(monthly_posts, temp_filename)
        print(f"[{tag}] Progress saved to {temp_filename}")

        # update current_ts to the earliest timestamp in the batch
        current_ts = min((post['timestamp'] for post in filtered_posts), default=0)

        #time.sleep(1)

    return monthly_posts


In [ ]:
for tag in tags:
    print(f"\n=== Starting fetch for tag: #{tag} ===")
    data_folder = f"data_{tag}"
    os.makedirs(data_folder, exist_ok=True)

    for year in range(2013, 2024):
        for month in range(1, 13):
            start_date = datetime(year, month, 1)
    
            # get the last day of the month
            if month == 12:
                end_date = datetime(year + 1, 1, 1) - timedelta(seconds=1)
            else:
                end_date = datetime(year, month + 1, 1) - timedelta(seconds=1)

            oldest_ts = int(start_date.timestamp())
            latest_ts = int(end_date.timestamp())
            month_key = start_date.strftime("%Y-%m")  # e.g., "2014-01"
    
            final_filename = os.path.join(data_folder, f"{month_key}.json")
            temp_filename = os.path.join(data_folder, f"{month_key}-temp.json")

            # if the final file exists, skip fetching for this month
            if os.path.exists(final_filename):
                print(f"[{tag}] Skipping {month_key}, data already fully fetched.")
                continue

            print(f"[{tag}] Fetching posts for {start_date.strftime('%B %Y')}...")

            posts = fetch_posts_for_period(oldest_ts, latest_ts, temp_filename)

            # if successful, rename the temp file to the final file 
            if posts:
                os.rename(temp_filename, final_filename)
                print(f"[{tag}] Saved {len(posts)} posts to {final_filename}")
            else:
                os.rename(temp_filename, final_filename)
                print(f"[{tag}] No posts found for {month_key}, empty file.")
    
        print(f"[{tag}] Finished fetching for year {year}. Moving to the next year...")

    print(f"[{tag}] Finished fetching and saving all posts!")

print("Done fetching posts for all tags!")